# Working with Custom Shape Libraries

When working with the `ipydrawio` TypeScript API, custom shape libraries can be added by providing a fully-resolved, absolute URL to an `.xml` file in `clibs`. From the Widget API, this is somewhat more complicated, but might be worth it for certain custom cases such as [issue #80](https://github.com/deathbeds/ipydrawio/issues/80).

In [ ]:
import base64, json, urllib.parse, zlib
import ipydrawio

## A Library
A library is, at its core, a list of shape descriptions.

In [ ]:
library = [{'w': 80,
  'h': 80,
  'aspect': 'fixed',
  'title': 'Source',
  'xml': '<mxGraphModel><root><mxCell id="0"/><mxCell id="1" parent="0"/><object label="" type="Source" interArrivalTime="" id="2"><mxCell style="rhombus;whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry width="80" height="80" as="geometry"/></mxCell></object></root></mxGraphModel>'},
 {'w': 80,
  'h': 80,
  'aspect': 'fixed',
  'title': 'Queue',
  'xml': '<mxGraphModel><root><mxCell id="0"/><mxCell id="1" parent="0"/><object label="" type="Queue" capacity="" id="2"><mxCell style="ellipse;whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry width="80" height="80" as="geometry"/></mxCell></object></root></mxGraphModel>'},
 {'w': 80,
  'h': 80,
  'aspect': 'fixed',
  'title': 'Machine',
  'xml': '<mxGraphModel><root><mxCell id="0"/><mxCell id="1" parent="0"/><object label="" type="Machine" processingTime="" id="2"><mxCell style="whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry width="80" height="80" as="geometry"/></mxCell></object></root></mxGraphModel>'},
 {'w': 80,
  'h': 80,
  'aspect': 'fixed',
  'title': 'Exit',
  'xml': '<mxGraphModel><root><mxCell id="0"/><mxCell id="1" parent="0"/><object label="" type="Exit" id="2"><mxCell style="rhombus;whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry width="80" height="80" as="geometry"/></mxCell></object></root></mxGraphModel>'}]

## JSON Encoding

Each element in the list has an `xml` attribute which is, as it suggests, XML, and must be carefully escaped. Because this will go through a URL parser, an XML parser, _and_ a JSON parser (and then another XML parser), the recommended approach is to use drawio's semi-convoluted base64/zlib technique.

In [ ]:
zlib_opts = dict(wbits=-15)

def inflate(deflated):
    infl = zlib.decompressobj(**zlib_opts)
    return urllib.parse.unquote(infl.decompress(base64.b64decode(deflated)) + infl.flush())

def deflate(inflated):
    defl = zlib.compressobj(**zlib_opts)
    return base64.b64encode(defl.compress(urllib.parse.quote(inflated).encode("utf-8")) + defl.flush()).decode('utf-8')

In addition, it's best to avoid extra spaces in the data URI.

In [ ]:
library_json = json.dumps(
    [dict(shape.items(), xml=deflate(shape["xml"])) for shape in library],
    separators=(",", ":")
)

This is wrapped inside an XML document, with a top-level tag of `mxlibrary`.

## XML Encoding

In [ ]:
library_xml = f"""<mxlibrary>{library_json}</mxlibrary><!-- /my library -->"""
library_xml

This, in turn, must be transformed into a [Data URI](https://developer.mozilla.org/en-US/docs/Web/HTTP/Basics_of_HTTP/Data_URIs). It can't be `base64` encoded, because drawio expects a semicolon-separated list of ids. 

> Use of data URIs relies on an **UNRELEASED PATCH** applied when packaging `@deathbeds/jupyterlab-drawio-webpack`: by default, the upstream would rewrite this into a proxied request, which `ipydrawio` don't support. It's also expecting to find the filename in the URL, hence the weird comment at the end.

## URL Encoding

In [ ]:
library_data_uri = f"data:application/xml,{library_xml}"

Finally, we can use this data URI to update `clibs` in `url_params`. The `U` is for `URL`.

> Some others that might be worth exploring some time include `L` for `Local`, which works with an `IndexedDB` instance... but is not guaranteed to be configured by the time a document loads.

### URL Params

In [ ]:
url_params = dict(ipydrawio.Diagram._default_url_params(None))
url_params.update(clibs=f"U{library_data_uri}", stealth="0", ui="min")
url_params

Note, we also override the `stealth` default... `stealth` isn't _strictly_ going to worsen the privacy posture, as all of the other providers are still disabled.

## The Widget

In [ ]:
d = ipydrawio.Diagram(url_params=url_params, layout=dict(height="800px"))
d

We should now be able to use the desired shapes in the diagram.

In [ ]:
canned_value = '<mxfile version="15.8.7" type="embed"><diagram id="x" name="My Diagram"><mxGraphModel dx="1687" dy="681" grid="1" gridSize="10" guides="1" tooltips="1" connect="1" arrows="1" fold="1" page="1" pageScale="1" pageWidth="850" pageHeight="1100" math="0" shadow="0"><root><mxCell id="0"/><mxCell id="1" parent="0"/><object label="a" type="Source" interArrivalTime="" id="2"><mxCell style="rhombus;whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry x="100" y="90" width="80" height="80" as="geometry"/></mxCell></object><object label="b" type="Queue" capacity="" id="3"><mxCell style="ellipse;whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry x="250" y="90" width="80" height="80" as="geometry"/></mxCell></object><object label="c" type="Machine" processingTime="" id="4"><mxCell style="whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry x="385" y="90" width="80" height="80" as="geometry"/></mxCell></object><object label="d" type="Exit" id="5"><mxCell style="rhombus;whiteSpace=wrap;html=1;" vertex="1" parent="1"><mxGeometry x="510" y="90" width="80" height="80" as="geometry"/></mxCell></object></root></mxGraphModel></diagram></mxfile>'
d.source.value = canned_value